> **Attribution:** Adapted from [Keithland89/StudioLens-for-Copilot-Studio](https://github.com/Keithland89/StudioLens-for-Copilot-Studio), licensed under the MIT License.
>
> Adapted by the ESS Insights team for ESS-only scoping.

# Copilot Credit Consumption — Ingester (Fabric)

Lands the three **Power Platform admin center Copilot Credits reports** (`EntitlementConsumption*_MCSMessages_*.csv`) into Lakehouse Delta tables for observed aggregate credit reporting.

The notebook reads files already landed in the Lakehouse; it does not acquire [Power Platform admin center](https://admin.powerplatform.microsoft.com) exports. Upload all three files manually to `Files/credit_consumption`, or implement a separate landing pipeline. No companion Power Automate flow is included in this repository.

```
Power Platform admin center  ─(download)─▶  manual/automated landing  ─▶  Lakehouse Files/credit_consumption/*.csv  ─▶  THIS notebook  ─▶  Delta
```

| Source CSV (prefix) | → Delta table | Grain |
| --- | --- | --- |
| `EntitlementConsumptionTenantDetailsReport_MCSMessages` | `dbo.credit_consumption_tenant` | environment × capacity × usage date (daily) |
| `EntitlementConsumptionTenantPerAgentDetailsReport_MCSMessages` | `dbo.credit_consumption_agent` | agent × feature × environment × lookback snapshot |
| `EntitlementConsumptionTenantPerUserDetailsReport_MCSMessages` | `dbo.credit_consumption_user` | user × agent × lookback snapshot |

**Fail-safe ingestion** — all three reports are required. Missing, duplicate, header-only, or malformed inputs fail during preflight before any Delta table is overwritten, so a failed run cannot erase or mix prior valid snapshots.

**Multi-environment** — each tenant-level Power Platform admin center export already carries all environments and the Agent / Tenant reports include `Environment Id` / `Environment Name`. Keep exactly one current file per report type in `Files/credit_consumption/`; multiple overlapping snapshots are rejected to prevent double counting.

**No app registration / Graph permission required** — this notebook only reads files already in the Lakehouse.

**Attribution boundary** — the Power Platform admin center exports have no conversation/session ID. Observed credits can be linked to ESS at bare `Agent Id` and, when nonblank, normalized `User Email`; any per-conversation, resolved-conversation, topic, or outcome value is a modeled allocation and must be labeled as such. Blank `User Email` rows remain in the user table as unmatched identity.

## 1. Configuration

Tag this cell as the pipeline **`parameters`** cell (cell ⋯ menu → *Toggle parameter cell*) if your Fabric orchestration needs to override `WRITE_MODE`. Output tables are schema-qualified (`dbo.`) to work with both schema-enabled and legacy Lakehouses.

In [ ]:
# === CONFIG ===  (tag this cell as the pipeline `parameters` cell)

# Folder in the attached Lakehouse where the three Power Platform admin center CSVs have been landed.
SOURCE_DIR  = 'Files/credit_consumption'

# Filename globs -> target Delta table. The default Microsoft export names are matched
# case-insensitively and tolerate the trailing _30 / _180 / (1) day-window + dedupe suffixes.
REPORTS = {
    'dbo.credit_consumption_tenant': 'EntitlementConsumptionTenantDetailsReport_MCSMessages*',
    'dbo.credit_consumption_agent':  'EntitlementConsumptionTenantPerAgentDetailsReport_MCSMessages*',
    'dbo.credit_consumption_user':   'EntitlementConsumptionTenantPerUserDetailsReport_MCSMessages*',
}

WRITE_MODE  = 'overwrite'   # 'overwrite' = current rolling snapshot;
                            # 'append' = snapshot history only. Overlapping lookbacks are not transactions;
                            # measures must select/deduplicate by LoadDate rather than sum snapshots.

ADD_LINEAGE = True          # add SourceFile + LoadDate columns to every row
STRICT      = True          # all three current, nonempty reports are required

## 2. Helpers

`_list_matches` resolves a glob against the Lakehouse `Files/` area (via `notebookutils.fs.ls` (mount-independent)). `_read_csv` parses with multiline + quote-escape so the reports' free-text columns (knowledge sources, scenario names) survive. `_sanitize` makes column names Delta-safe while the PBIP keeps reading the original names through Delta column-mapping.

In [ ]:
import os, glob, re, datetime
from pyspark.sql import functions as F

_LOAD_TS = datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

# Local POSIX mount for the attached Lakehouse: 'Files/x' -> '/lakehouse/default/Files/x'
def _local(path):
    return path if path.startswith('/') else f'/lakehouse/default/{path}'

def _list_matches(pattern):
    rx = re.compile('^' + re.escape(pattern).replace('\\*', '.*') + '$', re.IGNORECASE)
    try:
        entries = notebookutils.fs.ls(SOURCE_DIR)
    except Exception:
        return []
    hits = [e.name for e in entries if (not e.isDir) and rx.match(e.name) and e.name.lower().endswith('.csv')]
    return [f'{SOURCE_DIR}/{f}' for f in sorted(hits)]

def _read_csv(path):
    return (spark.read
            .option('header', True)
            .option('multiLine', True)
            .option('escape', '"')
            .option('encoding', 'UTF-8')
            .csv(path))

_INVALID = re.compile(r'[ ,;{}()\n\t=/-]')
def _sanitize(df):
    return df.toDF(*[_INVALID.sub('_', c.lstrip('\ufeff')) for c in df.columns])

print(f'Load timestamp: {_LOAD_TS}')
print(f'Source folder : {SOURCE_DIR}  (exists: {os.path.isdir(_local(SOURCE_DIR))})')

## 3. Ingest each report → Delta

Loops the `REPORTS` map. For each target table it requires exactly one nonempty current CSV, validates required headers, explicitly casts quantitative fields to `decimal(38,6)`, adds lineage columns, sanitises, and materialises the row count. Only after all three reports pass does it write any Delta table.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DecimalType

# Canonical (sanitised) schemas — used to build an empty table when a report is absent,
# so downstream PBIP columns always resolve.
EMPTY_SCHEMAS = {
    'dbo.credit_consumption_tenant': ['BillingPlan_Id','BillingPlan_Name','Environment_Id','Environment_Name',
        'Capacity_Type','Entitled_Quantity','Prepaid_Consumed_Quantity','Pay_as_you_go_Consumed_Quantity','Usage_Date'],
    'dbo.credit_consumption_agent': ['Agent_Name','Agent_Id','Product','AI_Feature_Billable_Feature','Billed_credit',
        'Non_billed_credit','Channel','Knowledge_Sources','Tool_Used','LLM_Model','Scenario_Name','Environment_Id','Environment_Name'],
    'dbo.credit_consumption_user': ['User_Id','User_Email','Agent_Id','Agent_Name','Billable_credit_used',
        'Credits_used','M365_Copilot_Licensed'],
}

NUMERIC_COLUMNS = {
    'dbo.credit_consumption_tenant': ['Entitled_Quantity','Prepaid_Consumed_Quantity','Pay_as_you_go_Consumed_Quantity'],
    'dbo.credit_consumption_agent': ['Billed_credit','Non_billed_credit'],
    'dbo.credit_consumption_user': ['Billable_credit_used','Credits_used'],
}

def _empty(cols):
    cols = list(cols)
    if ADD_LINEAGE:
        cols = cols + ['SourceFile', 'LoadDate']
    numeric = {c for values in NUMERIC_COLUMNS.values() for c in values}
    return spark.createDataFrame([], StructType([
        StructField(c, DecimalType(38, 6) if c in numeric else StringType(), True) for c in cols
    ]))

def _validate_and_cast(df, table, path):
    missing = sorted(set(EMPTY_SCHEMAS[table]) - set(df.columns))
    if missing:
        raise ValueError(
            f'{table}: matched file "{path}" is malformed; missing required column(s) after sanitization: '
            + ', '.join(missing)
        )
    for col in NUMERIC_COLUMNS[table]:
        df = df.withColumn(col, F.col(col).cast(DecimalType(38, 6)))
    return df

def _write(df, table):
    (df.write.mode(WRITE_MODE)
        .option('overwriteSchema', 'true')
        .option('delta.columnMapping.mode', 'name')
        .option('delta.minReaderVersion', '2')
        .option('delta.minWriterVersion', '5')
        .format('delta').saveAsTable(table))

if WRITE_MODE == 'append':
    print('⚠  APPEND stores snapshot history: overlapping rolling windows are not transactions. '
          'Use LoadDate-aware selection/deduplication and never sum snapshots blindly.')

# Preflight every report before writing anything, so a later failure cannot leave mixed snapshots.
prepared = {}
for table, pattern in REPORTS.items():
    matches = _list_matches(pattern)
    if not matches:
        raise FileNotFoundError(
            f'{table}: no file matched "{pattern}". All three current Power Platform admin center exports are required; '
            'no existing Delta table was changed.'
        )

    if len(matches) > 1:
        raise ValueError(
            f'{table}: {len(matches)} files matched "{pattern}". Keep exactly one current tenant-level '
            'snapshot per report type in Files/credit_consumption; overlapping Power Platform admin center lookback exports '
            'cannot be unioned without double counting. Matched files: ' + ', '.join(matches)
        )

    path = matches[0]
    df = _validate_and_cast(_sanitize(_read_csv(path)), table, path)
    if ADD_LINEAGE:
        df = df.withColumn('SourceFile', F.lit(os.path.basename(path))) \
               .withColumn('LoadDate',   F.lit(_LOAD_TS))

    n = df.count()
    if n == 0:
        raise ValueError(
            f'{table}: matched file "{path}" has headers but no data rows. '
            'No existing Delta table was changed.'
        )
    prepared[table] = (df, 1, n)

print('✓  Preflight passed for every report; starting Delta writes.')
summary = []
for table in REPORTS:
    df, files, n = prepared[table]
    _write(df, table)
    summary.append((table, files, n))
    print(f'✓  {table:34} — 1 current snapshot, {n:,} rows -> written ({WRITE_MODE})')

print('\nDone. Credit-consumption Delta tables written. Refresh the supplied Import model through its SQL analytics endpoint to pick them up.')

## 4. Verify + quick reconciliation

Prints observed Power Platform admin center headline totals and, if `agent_sessions` exists, the transcript-native diagnostic total side by side. This is not an exact reconciliation: the export has no conversation/session ID, and per-conversation or resolved-conversation attribution is modeled only.

In [ ]:
def _num(col):
    return F.sum(F.coalesce(F.col(col).cast('double'), F.lit(0.0)))

for table, files, rows in summary:
    print(f'{table:34} files={files} rows={rows:,}')

try:
    agent = spark.table('dbo.credit_consumption_agent')
    if agent.count():
        tot = agent.select(_num('Billed_credit').alias('billed'),
                           _num('Non_billed_credit').alias('nonbilled')).collect()[0]
        print(f"\nObserved Power Platform admin center credits — billed: {tot['billed']:,.2f}   non-billed: {tot['nonbilled']:,.2f}")
except Exception as e:
    print('agent consumption summary skipped:', e)

try:
    user = spark.table('dbo.credit_consumption_user')
    unmatched = user.filter(F.col('User_Email').isNull() | (F.trim(F.col('User_Email')) == '')).count()
    print(f'User rows with unmatched identity (blank User Email), retained: {unmatched:,}')
except Exception as e:
    print('user identity summary skipped:', e)

try:
    sess = spark.table('dbo.agent_sessions')
    disp = sess.select(_num('total_displayed_cost').alias('disp')).collect()[0]['disp']
    print(f"Transcript diagnostic — displayedCost credits: {disp:,.2f}")
    print('(Diagnostic comparison only; the Power Platform admin center export supplies no conversation/session ID. Do not treat this as exact billing attribution.)')
except Exception:
    print('(agent_sessions not present yet — run the transcript parser to enable reconciliation.)')

---
**Connect the PBIP**: these three tables feed the semantic-model queries `Credit Consumption (Agent)`, `Credit Consumption (User)`, `Credit Consumption (Tenant)`, all gated by the `Enable_Consumption` parameter. Leave `Enable_Consumption = false` for customers who don't supply the reports and the billing visuals stay dormant — the transcript-native `Total Cost Units` measure keeps working regardless.